In [13]:
import math

### Objects

In [1]:
class Shape: 
    def __init__(self, name):
        self.name = name
    
    def perimeter(self):
        raise NotImplementedError("perimeter not yet implemented")
    
    def area(self):
        raise NotImplementedError("area not yet implemented")

In [10]:
class Square(Shape):
    def __init__(self, name, side):
        super().__init__(name)
        self.side = side
    
    def perimeter(self):
        return 4 * self.side
    
    def area(self):
        return self.side ** 2 
    

In [14]:
class Circle(Shape):
    def __init__(self, name, radius):
        super().__init__(name)
        self.radius = radius
    
    def perimeter(self):
        return 2 * math.pi * self.radius

    def area(self):
        return math.pi * self.radius ** 2

In [26]:
examples = [Square("sq", 3), Circle("ci", 2)]
for item in examples:
    n = item.name
    p = item.perimeter()
    a = item.area()
    print(f"{n} has a perimeter {p:.2f} and area {a:.2f}")

sq has a perimeter 12.00 and area 9.00
ci has a perimeter 12.57 and area 12.57


In [19]:
# When Python executes the code below, it creates an object in memory 
# that contains the instructions to print a string and assigns that object to the variable example:
def example():
    print("in example")
    
example()
alias = example
alias()

in example
in example


We can also store function objects in data structures like lists and dictionaries.

In [ ]:
def square_perimeter(thing):
    return 4*thing["side"]

def square_area(thing):
    return thing["side"]**2

def square_new(name, side):
    return {
        "name": name, 
        "side": side,
        "perimeter": square_perimeter,
        "area": square_area
    }

In [23]:
def circle_perimeter(thing):
    return 2 * math.pi * thing["radius"]

def circle_area(thing):
    return math.pi * thing["radius"]**2

def circle_new(name, radius):
    return {
        "name": name, 
        "radius": radius,
        "perimeter": circle_perimeter,
        "area": circle_area
    }

In [28]:
def call(thing, method_name):
    return thing[method_name](thing)

examples = [square_new("sq", 3), circle_new("ci", 2)]
for item in examples:
    n = item['name']
    p = call(item, "perimeter")
    a = call(item, "area")
    print(f"{n} {p:.2f} {a:.2f}")

sq 12.00 9.00
ci 12.57 12.57


The function `call` looks up the function stored in the dictionary, then calls that function with the dictionary as its first object; in other words, instead of using `obj.meth(arg)` we use `obj["meth"](obj, arg)`. Behind the scenes, this is (almost) how objects actually work. We can think of an object as a special kind of dictionary. A method is just a function that takes an object of the right kind as its first parameter (typically called `self` in Python).

### Classes
One problem with implementing objects as dictionaries is that it allows every single object to behave slightly differently. In practice, we want objects to store different values (e.g., different squares to have different sizes) but the same behaviors (e.g., all squares should have the same methods). 

In [31]:
Square = {
    "perimeter": square_perimeter,
    "area": square_area,
    "_classname" :"Square"
}

def square_new(name, side):
    return {
        "name": name, 
        "side": side, 
        "_class": Square
    }

In [32]:
Circle = {
    "perimeter": circle_perimeter,
    "area": circle_area,
    "_classname" :"Circle"
}

def circle_new(name, radius):
    return {
        "name": name, 
        "radius": radius, 
        "_class": Circle
    }

In [33]:
def call(thing, method_name):
    return thing["_class"][method_name](thing)

examples = [square_new("sq", 3), circle_new("ci", 2)]
for ex in examples:
    n = ex["name"]
    p = call(ex, "perimeter")
    a = call(ex, "area")
    c = ex["_class"]["_classname"]
    print(f"{n} is a {c}: {p:.2f} {a:.2f}")

sq is a Square: 12.00 9.00
ci is a Circle: 12.57 12.57


### Arguments 

> **Arguments vs. Parameters**
     Many programmers use the words argument and parameter interchangeably, but to make our meaning clear, we call the values passed into a function its arguments and the names the function uses to refer to them as its parameters. Put another way, parameters are part of the definition, and arguments are given when the function is called.

Python gives us a better way. If we define a parameter in a function with a leading *, it captures any "extra" values passed to the function that don't line up with named parameters. Similarly, if we define a parameter with two leading stars **, it captures any extra named parameters:

In [41]:
def show_args(title, *args, **kwargs):
    print(f"{title} args '{args}' and kwargs '{kwargs}'")
    
show_args("nothing")
show_args("one unnamed argument", 1)
show_args("one named argument", second="2")
show_args("one of each", 3, fourth="4")

nothing args '()' and kwargs '{}'
one unnamed argument args '(1,)' and kwargs '{}'
one named argument args '()' and kwargs '{'second': '2'}'
one of each args '(3,)' and kwargs '{'fourth': '4'}'


A complementary mechanism called spreading allows us to take a list or dictionary full of arguments and spread them out in a call to match a function's parameters:

In [43]:
def show_spread(left, middle, right):
    print(f"left {left} middle {middle} and right {right}")

all_in_list = [1,2,3]
show_spread(*all_in_list)

all_in_dict = {"right":30 , "left":10, "middle":20}
show_spread(**all_in_dict)

left 1 middle 2 and right 3
left 10 middle 20 and right 30


In [53]:
def square_larger(thing,size):
    return call(thing, "area") > size

Square = {
    "perimeter": square_perimeter, 
    "area": square_area, 
    "larger": square_larger, 
    "_classname": Square
}

def square_new(name, side):
    return {
        "name":name, 
        "side": side, 
        "_class":Square
    }

def circle_larger(thing, size):
    return call(thing, "area") > size

Circle = {
    "perimeter": circle_perimeter, 
    "area": circle_area, 
    "larger": circle_larger, 
    "_classname": Circle
}

def circle_new(name, radius):
    return {
        "name":name, 
        "radius": radius, 
        "_class":Circle
    }

def call(thing, method_name, *args):
    return thing["_class"][method_name](thing, *args)


In [55]:
examples = [square_new("sq",3), circle_new('ci', 2)]
for ex in examples: 
    result = call(ex, "larger", 10)
    print(f"is {ex['name']} larger? {result}")

is sq larger? False
is ci larger? True


## Core Lessons
1. As systems get larger, their complexity grows disproportionately, not linearly. The primary goal of software design is to break a large system into small pieces that interact in very few, well-defined ways.
